In [0]:
%run ../00_common/data_utils

In [0]:
%run ../00_common/UDF_utils

#0. Base

In [0]:
# ==============================
# 市场特性配置（列级别条件 — 所有市场在同一批次中混合处理）
# ==============================
MARKETS_DISABLE_ADDRESS_PATH4 = ["KOR", "TWN"]     # 差异1: 禁用地址匹配 Path4

# OtherBlockingKey
MARKETS_DISABLE_BLOCKING_KEY  = []                 # 差异2: 无 OtherBlockingKey (始终 Normal)
MARKETS_ENABLE_DRJART         = ["KOR"]            # 差异2: DrJart 品牌43

MARKETS_ENABLE_UNBIND         = ["JPN", "THA"]     # 差异3: UNBIND 过滤
MARKETS_ENABLE_LINE_MEDIA     = ["JPN"]            # 差异4: LINE 媒体类型

MARKETS_ENABLE_LOCALNAME2     = ["JPN"]            # 差异5: 启用local name 2 进行match


# 差异6: 是否启用中文排除名单
MARKETS_ENABLE_CHINESE_EXCLUDED = ["TWN"]


# 名字排除名单, 应用所有market（匹配 scon_localfullname / scon_localfirstname，在排除名单中的master表的名字不参与match）
EXCLUDED_LOCAL_NAMES = [
    "DO NOT", "DO NOT SHIP", "TEST", "TESTTEST", "TEST TEST", "DO  NOT"
]

# 中文名字排除名单, 应用 MARKETS_ENABLE_CHINESE_EXCLUDED 中market
EXCLUDED_LOCAL_NAMES_WITH_CHINESE = [
    "不", "留"
]

In [0]:
# ==============================
# 公共函数
# ==============================

def is_scon_name_empty(tab_alias: str) -> Column:
    """判断 master 姓名字段是否全空（8个字段）"""
    cols = [
        f"{tab_alias}.scon_englishfirstname", f"{tab_alias}.scon_englishmiddlename",
        f"{tab_alias}.scon_englishlastname",  f"{tab_alias}.scon_englishfullname",
        f"{tab_alias}.scon_localfirstname",   f"{tab_alias}.scon_localmiddlename",
        f"{tab_alias}.scon_locallastname",    f"{tab_alias}.scon_localfullname"
    ]
    cond = F.lit(True)
    for c in cols:
        cond = cond & (F.col(c).isNull() | (F.trim(F.col(c)) == ""))
    return cond


def fallback_to_src(con_col: str, src_col: str) -> Column:
    """master 字段为空时回退到 source 字段"""
    return (
        F.when(F.col(con_col).isNull() | (F.trim(F.col(con_col)) == ""),
               F.coalesce(F.col(src_col), F.lit("")))
         .otherwise(F.col(con_col))
    )


def build_other_blocking_key(mrkt_col: str, brnd_col: str, grp_col: str) -> Column:
    """
    差异 2: 构建 OtherBlockingKey — 列级别市场分支
    - PHL: 始终 'Normal'（无 blocking key）
    - KOR Brand43: 'DrJart'
    - Brand99 + empl: 'EmpBrand99'
    - 其余: 'Normal'
    """
    _mrkt = F.col(mrkt_col)
    return (
        F.when(_mrkt.isin(MARKETS_DISABLE_BLOCKING_KEY), F.lit("Normal"))
         .when(_mrkt.isin(MARKETS_ENABLE_DRJART) &
               (F.coalesce(F.col(brnd_col), F.lit("")) == "43"),
               F.lit("DrJart"))
         .when((F.coalesce(F.col(brnd_col), F.lit("")) == "99") &
               (F.coalesce(F.col(grp_col), F.lit("")) == "empl"),
               F.lit("EmpBrand99"))
         .otherwise(F.lit("Normal"))
    )

In [0]:
# ==============================
# 排除名单（黑名单）— 全部带 MarketCode
# ==============================
def get_exclude_dfs():

    t_merge_exclude_phone_config = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_phone_config")
        .select(
            F.col("MarketCode").alias("exclude_mrkt"),
            F.col("phoneNumber").alias("phone"),
        )
    )
    t_merge_exclude_media_config = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_media_config")
        .select(
            F.col("MarketCode").alias("exclude_mrkt"),
            F.col("mediaAddress").alias("email"),
        )
    )
    t_merge_exclude_address_config = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_address_config")
        .select(
            F.col("MarketCode").alias("exclude_mrkt"),
            F.col("Address1").alias("addr"),
        )
    )

    # tmatchexcludeconsumer — 排除特定源系统(ACS和LineBind)
    t_merge_exclude_consumer_config = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config")
        .filter(F.col("tmec_type").isin(SOURCE_TYPE_ACS, SOURCE_TYPE_LINEBIND))
        .select(
            F.col("tmec_marketcode").alias("exclude_mrkt"),
            F.col("tmec_sourcesystemcode").alias("exclude_srcs_code"),
        )
        .distinct()
    )

    # master 侧用的排除表（需要不同别名避免 left_anti 列冲突）
    exclude_phones = t_merge_exclude_phone_config.select(
        F.col("exclude_mrkt").alias("exclude_phone_mrkt"),
        F.col("phone").alias("exclude_phone"),
    ).distinct()

    exclude_emails = t_merge_exclude_media_config.select(
        F.col("exclude_mrkt").alias("exclude_email_mrkt"),
        F.col("email").alias("exclude_email"),
    ).distinct()

    return t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config, t_merge_exclude_consumer_config, exclude_phones, exclude_emails

#1.1 batch recode to vertex

In [0]:
# ==============================
# 1.1 构建 batch profile（output1_df）
# ==============================

def get_batch_data(task_id, t_merge_exclude_consumer_config, t_merge_exclude_phone_config, t_merge_exclude_media_config, t_merge_exclude_address_config):

    # 替换cid
    cid_mapping_df = (get_cidGroup_by_process(task_id)
            .where(f"TASK_ID = '{task_id}' AND record_type = '{CID_MATCH_RECORD_TYPE_BATCH}'")
            .select("mrkt_code", "srcc_id", "new_mapping_conusmer_id")
            .distinct())

    # source 侧排除 tmatchexcludeconsumer + MarketCode
    t_clean_consumer = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer").alias("tcc")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("is_include") == True)
        .join(t_merge_exclude_consumer_config,
            (F.col("srcc_srcs_code") == F.col("exclude_srcs_code")) &
            (F.col("srcc_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
        .join(cid_mapping_df.alias("cmd"),
                (F.col("tcc.srcc_id") == F.col("cmd.srcc_id")) &
                (F.col("tcc.srcc_mrkt_code") == F.col("cmd.mrkt_code")),
                "left")
        .select(
            "tcc.*",
            "cmd.new_mapping_conusmer_id"
        )
        .withColumn("SRCC_CONSUMERID", F.when(F.col("cmd.new_mapping_conusmer_id").isNotNull(), F.col("cmd.new_mapping_conusmer_id")).otherwise(F.col("tcc.SRCC_CONSUMERID")))
        .withColumn("is_master_recode", F.lit(False))
        .withColumn("master_recode_create_time", F.lit(None).cast(TimestampType()))
        .drop("new_mapping_conusmer_id")
        
    )


    # 全 market 电话排除 + MarketCode
    clean_phone_vld = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_phone")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("SRCP_QUALITY_CODE") == "vld")
        .filter(F.trim(F.col("srcp_phonenumber")) != "")
        .join(t_merge_exclude_phone_config,
            (F.col("srcp_phonenumber") == F.col("phone")) &
            (F.col("srcp_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
    )


    # 全 market 邮件排除(JPN LINE 媒体扩展)
    clean_emedia_vld = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_emedia")
        .filter(F.col("task_id") == task_id)
        .filter(
            (F.col("SRCE_QUALITY_CODE") == "vld") |
            (F.col("srce_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("srce_emdt_code") == "scllneprs"))
        )
        .filter(F.trim(F.col("srce_address")) != "")
        .join(t_merge_exclude_media_config,
            (F.col("srce_address") == F.col("email")) &
            (F.col("srce_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
        .withColumn("srce_address", F.lower(F.col("srce_address")))
    )


    # + MarketCode
    clean_address_all = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_address")
        .filter(F.col("task_id") == task_id)
        .filter(F.trim(F.coalesce(F.col("srca_address1"), F.lit(""))) != "")
        .join(t_merge_exclude_address_config,
            (F.col("srca_address1") == F.col("addr")) &
            (F.col("srca_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
    )


    t_clean_consumergroup = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumergroup")
        .filter(F.col("task_id") == task_id)
    )

    output1_df = (
        t_clean_consumer.alias("a")
        .join(clean_phone_vld.alias("b"),
            (F.col("b.SRCP_SRCC_ID") == F.col("a.SRCC_ID")) &
            (F.col("b.srcp_mrkt_code") == F.col("a.srcc_mrkt_code")),
            "left")
        .join(clean_emedia_vld.alias("c"),
            (F.col("c.SRCE_SRCC_ID") == F.col("a.SRCC_ID")) &
            (F.col("c.srce_mrkt_code") == F.col("a.srcc_mrkt_code")),
            "left")
        .join(clean_address_all.alias("d"),
            (F.col("d.SRCA_SRCC_ID") == F.col("a.SRCC_ID")) &
            (F.col("d.srca_mrkt_code") == F.col("a.srcc_mrkt_code")) &
            (~F.col("a.srcc_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4)),
            "left")
        .join(t_clean_consumergroup.alias("g"),
            (F.col("a.SRCC_ID") == F.col("g.SRCG_SRCC_ID")) &
            (F.col("g.SRCG_mrkt_code") == F.col("a.srcc_mrkt_code")),
            "left")
        .select(
            F.lit(None).cast("string").alias("scon_id"),
            F.lit(None).cast("string").alias("consumermdmkey"),

            F.col("a.srcc_id").alias("scon_srcc_id"),
            F.col("a.srcc_srcs_code").alias("scon_srcs_code"),
            F.col("a.srcc_sourcetimestamp").alias("scon_sourcetimestamp"),
            F.col("a.srcc_mrkt_code").alias("scon_mrkt_code"),
            F.col("a.srcc_brnd_code").alias("scon_brnd_code"),
            F.col("a.srcc_consumerid").alias("scon_consumerid"),

            F.col("a.srcc_englishfirstname").alias("scon_englishfirstname"),
            F.col("a.srcc_englishmiddlename").alias("scon_englishmiddlename"),
            F.col("a.srcc_englishlastname").alias("scon_englishlastname"),
            F.col("a.srcc_englishfullname").alias("scon_englishfullname"),

            F.col("a.srcc_localfirstname").alias("scon_localfirstname"),
            F.col("a.srcc_localmiddlename").alias("scon_localmiddlename"),
            F.col("a.srcc_locallastname").alias("scon_locallastname"),
            F.col("a.srcc_localfullname").alias("scon_localfullname"),

            F.col("a.srcc_localfirstname2").alias("scon_localfirstname2"),
            F.col("a.srcc_localmiddlename2").alias("scon_localmiddlename2"),
            F.col("a.srcc_locallastname2").alias("scon_locallastname2"),
            F.col("a.srcc_localfullname2").alias("scon_localfullname2"),

            F.coalesce(F.col("c.srce_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("b.srcp_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.trim(F.col("d.srca_address1")), F.lit("")).alias("scad_address1"),
            F.coalesce(F.trim(F.col("d.srca_address2")), F.lit("")).alias("scad_address2"),
            F.coalesce(F.trim(F.col("d.srca_address3")), F.lit("")).alias("scad_address3"),
            F.coalesce(F.trim(F.col("d.srca_city_localdesc")), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.trim(F.col("d.srca_postalcode")), F.lit("")).alias("scad_postalcode"),

            F.col("a.is_master_recode"),
            F.col("a.master_recode_create_time"),
            F.col("a.batch_id"),

            # 差异 2: OtherBlockingKey（PHL→Normal, KOR Brand43→DrJart, Brand99+empl→EmpBrand99）
            build_other_blocking_key(
                "a.srcc_mrkt_code",
                "a.srcc_brnd_code",
                "g.srcg_consumer_grp"
            ).alias("OtherBlockingKey"),
        ).distinct()
    )

    return output1_df, t_clean_consumer, clean_phone_vld, clean_emedia_vld, clean_address_all

#1.2 master recode to vertex

In [0]:
# -----------------------------
# Path 1: CID 匹配 — source INNER JOIN master by consumerid + brnd_code
# -----------------------------
def get_path1_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer):

    path1 = (
        t_clean_consumer
        .select("srcc_consumerid", "srcc_srcs_code", "srcc_brnd_code", "srcc_mrkt_code").distinct().alias("src")
        .join(
            t_master_consumer.alias("con"),
            (F.col("src.srcc_consumerid") == F.col("con.scon_consumerid")) &
            (F.col("src.srcc_srcs_code") == F.col("con.scon_srcs_code")) &
            (F.col("src.srcc_brnd_code") == F.col("con.scon_brnd_code")) &
            (F.col("src.srcc_mrkt_code") == F.col("con.scon_mrkt_code")),
            "inner"
        )
        .join(t_master_phone_vld.alias("ph"),
            (F.col("con.scon_id") == F.col("ph.scph_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ph.scph_mrkt_code")),
            "left")
        .join(t_master_emedia_vld.alias("me"),
            (F.col("con.scon_id") == F.col("me.scme_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("me.scme_mrkt_code")),
            "left")
        .join(t_master_address.alias("ad"),
            (F.col("con.scon_id") == F.col("ad.scad_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ad.scad_mrkt_code")) &
            (~F.col("con.scon_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4)),
            "left")
        .join(t_master_consumergroup.alias("grp"),
            (F.col("con.scon_id") == F.col("grp.scgr_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("grp.scgr_mrkt_code")),
            "left")
        .select(
            F.col("con.scon_id"),
            F.col("con.consumermdmkey"),
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.col("con.scon_consumerid"),
            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            F.coalesce(F.col("me.scme_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("ph.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.col("ad.scad_address1"), F.lit("")).alias("scad_address1"),
            F.coalesce(F.col("ad.scad_address2"), F.lit("")).alias("scad_address2"),
            F.coalesce(F.col("ad.scad_address3"), F.lit("")).alias("scad_address3"),
            F.coalesce(F.col("ad.scad_city_localdesc"), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.col("ad.scad_postalcode"), F.lit("")).alias("scad_postalcode"),
            F.col("con.is_master_recode"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id"),
            build_other_blocking_key("con.scon_mrkt_code", "con.scon_brnd_code", "grp.scgr_consumer_grp")
                .alias("OtherBlockingKey"),
        ).distinct()
    )

    return path1

In [0]:
# -----------------------------
# Path 2: Phone 匹配 — source phone INNER JOIN master phone → master consumer
# -----------------------------
def get_path2_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_phone_vld):
    path2 = (
        # t_clean_consumer.alias("src")
        # .join(clean_phone_vld.alias("sp"),
        #     (F.col("src.srcc_id") == F.col("sp.srcp_srcc_id")) &
        #     (F.col("src.srcc_mrkt_code") == F.col("sp.srcp_mrkt_code")),
        #     "inner")
        clean_phone_vld
        .select("srcp_mrkt_code", "srcp_phonenumber").distinct().alias("sp")
        .join(t_master_phone_vld.alias("cp"),
            (F.col("sp.srcp_phonenumber") == F.col("cp.scph_phonenumber")) &
            (F.col("sp.srcp_mrkt_code") == F.col("cp.scph_mrkt_code")),
            "inner")
        .join(t_master_consumer.alias("con"),
            (F.col("cp.scph_scon_id") == F.col("con.scon_id")) &
            (F.col("cp.scph_mrkt_code") == F.col("con.scon_mrkt_code")) &
            (~is_scon_name_empty("con")),
            "inner")
        .join(t_master_emedia_vld.alias("me"),
            (F.col("con.scon_id") == F.col("me.scme_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("me.scme_mrkt_code")),
            "left")
        .join(t_master_address.alias("ad"),
            (F.col("con.scon_id") == F.col("ad.scad_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ad.scad_mrkt_code")) &
            (~F.col("con.scon_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4)),
            "left")
        .join(t_master_consumergroup.alias("grp"),
            (F.col("con.scon_id") == F.col("grp.scgr_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("grp.scgr_mrkt_code")),
            "left")
        .select(
            F.col("con.scon_id"),
            F.col("con.consumermdmkey"),
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.col("con.scon_consumerid"),
            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            F.coalesce(F.col("me.scme_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("cp.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.col("ad.scad_address1"), F.lit("")).alias("scad_address1"),
            F.coalesce(F.col("ad.scad_address2"), F.lit("")).alias("scad_address2"),
            F.coalesce(F.col("ad.scad_address3"), F.lit("")).alias("scad_address3"),
            F.coalesce(F.col("ad.scad_city_localdesc"), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.col("ad.scad_postalcode"), F.lit("")).alias("scad_postalcode"),
            F.col("con.is_master_recode"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id"),
            build_other_blocking_key("con.scon_mrkt_code", "con.scon_brnd_code", "grp.scgr_consumer_grp")
                .alias("OtherBlockingKey"),
        ).distinct()
    )

    return path2

In [0]:
# -----------------------------
# Path 3: Email 匹配 — source emedia INNER JOIN master media → master consumer
# -----------------------------
def get_path3_data(t_master_consumer, t_master_phone_vld,  t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_emedia_vld):

    
    path3 = (
        # t_clean_consumer.alias("src")
        # .join(clean_emedia_vld.alias("se"),
        #     (F.col("src.srcc_id") == F.col("se.srce_srcc_id")) &
        #     (F.col("src.srcc_mrkt_code") == F.col("se.srce_mrkt_code")),
        #     "inner")
        clean_emedia_vld
        .select("srce_mrkt_code", "srce_address").distinct().alias("se")
        .join(
            # 差异 4: 基本 emdt_code 过滤 + JPN LINE 媒体类型
            t_master_emedia_vld.alias("cm"),
            (F.col("se.srce_address") == F.col("cm.scme_address")) &
            (F.col("se.srce_mrkt_code") == F.col("cm.scme_mrkt_code")),
            "inner")
        .join(t_master_consumer.alias("con"),
            (F.col("cm.scme_scon_id") == F.col("con.scon_id")) &
            (F.col("cm.scme_mrkt_code") == F.col("con.scon_mrkt_code")) &
            (~is_scon_name_empty("con")),
            "inner")
        .join(t_master_phone_vld.alias("ph"),
            (F.col("con.scon_id") == F.col("ph.scph_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ph.scph_mrkt_code")),
            "left")
        .join(t_master_address.alias("ad"),
            (F.col("con.scon_id") == F.col("ad.scad_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ad.scad_mrkt_code")) &
            (~F.col("con.scon_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4)),
            "left")
        .join(t_master_consumergroup.alias("grp"),
            (F.col("con.scon_id") == F.col("grp.scgr_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("grp.scgr_mrkt_code")),
            "left")
        # 注意：me_alias="cm"（来自 INNER JOIN 的匹配邮箱），ph_alias="ph"（补充 LEFT JOIN）
        .select(
            F.col("con.scon_id"),
            F.col("con.consumermdmkey"),
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.col("con.scon_consumerid"),
            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            F.coalesce(F.col("cm.scme_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("ph.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.col("ad.scad_address1"), F.lit("")).alias("scad_address1"),
            F.coalesce(F.col("ad.scad_address2"), F.lit("")).alias("scad_address2"),
            F.coalesce(F.col("ad.scad_address3"), F.lit("")).alias("scad_address3"),
            F.coalesce(F.col("ad.scad_city_localdesc"), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.col("ad.scad_postalcode"), F.lit("")).alias("scad_postalcode"),
            F.col("con.is_master_recode"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id"),
            build_other_blocking_key("con.scon_mrkt_code", "con.scon_brnd_code", "grp.scgr_consumer_grp")
                .alias("OtherBlockingKey"),
        ).distinct()
    )

    return path3

In [0]:
# -----------------------------
# Address 匹配 — source address INNER JOIN master address → master consumer
# KOR/TWN 的行通过 filter 排除，其余市场正常参与
# -----------------------------
def get_path4_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_address_all):

    # 子查询：source address × source consumer（去重 mrkt_code + address1 + address2）
    # _src_addr = (
    #     clean_address_all.alias("sa")
    #     .join(t_clean_consumer.alias("sc"),
    #         (F.col("sa.srca_srcc_id") == F.col("sc.srcc_id")) &
    #         (F.col("sa.srca_mrkt_code") == F.col("sc.srcc_mrkt_code")),
    #         "inner")

    #     .filter(F.regexp_replace(F.coalesce(F.col("sa.srca_address1"), F.lit("")), r"[\r\n]", "") != "")
    #     .select(
    #         F.col("sc.srcc_mrkt_code"),
    #         F.col("sa.srca_address1"),
    #         F.col("sa.srca_address2"),
    #     ).distinct()
    # )

    _src_addr = (
        clean_address_all.alias("sa")
        .filter(F.regexp_replace(F.coalesce(F.col("sa.srca_address1"), F.lit("")), r"[\r\n]", "") != "")
        .select(
            F.col("sa.srca_mrkt_code"),
            F.col("sa.srca_address1"),
            F.col("sa.srca_address2"),
        ).distinct()
    )

    path4 = (
        t_master_address.alias("mad")
        .join(_src_addr.alias("sad"),
            (F.col("sad.srca_address1") == F.col("mad.scad_address1")) &
            (F.col("sad.srca_address2") == F.col("mad.scad_address2")) &
            (F.col("mad.scad_mrkt_code") == F.col("sad.srca_mrkt_code")),
            "inner")
        .join(t_master_consumer.alias("con"),
            (F.col("mad.scad_scon_id") == F.col("con.scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("mad.scad_mrkt_code")) &
            (~is_scon_name_empty("con")),
            "inner")
        .join(t_master_emedia_vld.alias("me"),
            (F.col("con.scon_id") == F.col("me.scme_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("me.scme_mrkt_code")),
            "left")
        .join(t_master_phone_vld.alias("ph"),
            (F.col("con.scon_id") == F.col("ph.scph_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ph.scph_mrkt_code")),
            "left")
        .join(t_master_consumergroup.alias("grp"),
            (F.col("con.scon_id") == F.col("grp.scgr_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("grp.scgr_mrkt_code")),
            "left")
        .filter(~F.col("con.scon_mrkt_code").isin(MARKETS_DISABLE_ADDRESS_PATH4))
        .select(
            F.col("con.scon_id"),
            F.col("con.consumermdmkey"),
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.col("con.scon_consumerid"),
            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            F.coalesce(F.col("me.scme_address"), F.lit("")).alias("scme_address"),
            F.coalesce(F.col("ph.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.coalesce(F.col("mad.scad_address1"), F.lit("")).alias("scad_address1"),
            F.coalesce(F.col("mad.scad_address2"), F.lit("")).alias("scad_address2"),
            F.coalesce(F.col("mad.scad_address3"), F.lit("")).alias("scad_address3"),
            F.coalesce(F.col("mad.scad_city_localdesc"), F.lit("")).alias("scad_city_localdesc"),
            F.coalesce(F.col("mad.scad_postalcode"), F.lit("")).alias("scad_postalcode"),
            F.col("con.is_master_recode"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id"),
            build_other_blocking_key("con.scon_mrkt_code", "con.scon_brnd_code", "grp.scgr_consumer_grp")
                .alias("OtherBlockingKey"),
        ).distinct()
    )

    return path4

In [0]:
# ==============================
# master 数据
# ==============================
#  (UNBIND 过滤) 

def get_master_data(t_merge_exclude_consumer_config, exclude_phones, exclude_emails, t_merge_exclude_address_config, t_clean_consumer, clean_phone_vld, clean_emedia_vld, clean_address_all):

    t_master_consumer = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
        .join(t_merge_exclude_consumer_config,
            (F.col("scon_srcs_code") == F.col("exclude_srcs_code")) &
            (F.col("scon_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
        .filter(
            ~(F.col("scon_mrkt_code").isin(MARKETS_ENABLE_UNBIND) & (F.coalesce(F.col("scon_srcc_action"), F.lit("")) == "UNBIND"))
            & 
            ~(
                F.upper(F.coalesce(F.col("scon_localfullname"), F.lit(""))).isin(EXCLUDED_LOCAL_NAMES) | 
                F.upper(F.coalesce(F.col("scon_localfirstname"), F.lit(""))).isin(EXCLUDED_LOCAL_NAMES)
            )
            &
            ~(  
                F.col("scon_mrkt_code").isin(MARKETS_ENABLE_CHINESE_EXCLUDED) &
                (
                    F.upper(F.coalesce(F.col("scon_localfullname"), F.lit(""))).isin(EXCLUDED_LOCAL_NAMES_WITH_CHINESE) | 
                    F.upper(F.coalesce(F.col("scon_localfirstname"), F.lit(""))).isin(EXCLUDED_LOCAL_NAMES_WITH_CHINESE)
                )
            )
        )
        .withColumn("is_master_recode", F.lit(True))
        .withColumn("master_recode_create_time", F.col("scon_creation_dt"))
    )

    t_master_phone_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_phone")
        .filter(F.col("scph_quality_code") == "vld")
        .filter(F.coalesce(F.col("scph_phonenumber"), F.lit("")) != "")
        .join(exclude_phones,
            (F.col("scph_phonenumber") == F.col("exclude_phone")) &
            (F.col("scph_mrkt_code") == F.col("exclude_phone_mrkt")),
            "left_anti")
    )

    # (JPN LINE 媒体扩展) 
    t_master_emedia_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_emedia")
        .join(exclude_emails,
            (F.col("scme_address") == F.col("exclude_email")) &
            (F.col("scme_mrkt_code") == F.col("exclude_email_mrkt")),
            "left_anti")
        .filter(
            (F.col("scme_quality_code") == "vld") |
            (F.col("scme_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("scme_emdt_code") == "scllneprs"))
        )
        .withColumn("scme_address", F.lower(F.col("scme_address")))
    )

    t_master_address = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_address")
        .filter(F.trim(F.coalesce(F.col("scad_address1"), F.lit(""))) != "")
        .join(t_merge_exclude_address_config,
            (F.col("scad_address1") == F.col("addr")) &
            (F.col("scad_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti")
    )

    t_master_consumergroup = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer_group")
    )

    path1 = get_path1_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer)
    path2 = get_path2_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_phone_vld)
    path3 = get_path3_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_emedia_vld)
    path4 = get_path4_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_master_address, t_master_consumergroup, t_clean_consumer, clean_address_all)


    output2_df = (
        path1
        .unionByName(path2)
        .unionByName(path3)
        .unionByName(path4)
        .distinct()
    )
    
    return output2_df


#1.3 all record with match key

In [0]:
def generate_match_key(output_df):
    merged_df = (
        output_df
        .withColumn("EmailMatchKey", email_match_udf(F.col("scme_address")))
        .withColumn("PhoneMatchKey", phone_match_udf(F.col("scph_phonenumber")))
        .withColumn("AddressMatchKey",
                    address_match_udf(F.col("scad_address1"), F.col("scad_address2"), F.col("scad_address3"),F.col("scad_city_localdesc"), F.col("scad_postalcode")))
        
        .withColumn("EnglishLNameMatchKey", cleanse_name_udf(F.col("scon_englishlastname")))
        .withColumn("EnglishFNameMatchKey", cleanse_name_udf(F.col("scon_englishfirstname")))
        .withColumn("LocalLNameMatchKey", cleanse_name_udf(F.col("scon_locallastname")))
        .withColumn("LocalFNameMatchKey", cleanse_name_udf(F.col("scon_localfirstname")))
        .withColumn("LocalLNameMatchKey2", cleanse_name_udf(F.col("scon_locallastname2"))) 
        .withColumn("LocalFNameMatchKey2", cleanse_name_udf(F.col("scon_localfirstname2")))

        .withColumn("name_key", 
            F.when(F.col("scon_mrkt_code").isin(MARKETS_ENABLE_LOCALNAME2), 
                full_name_sort_match_key_v3_udf(F.col("scon_englishfirstname"), F.col("scon_englishlastname"), F.col("scon_englishfullname"),
                                                F.col("scon_localfirstname"), F.col("scon_locallastname"), F.col("scon_localfullname"),
                                                F.col("scon_localfirstname2"), F.col("scon_locallastname2"), F.col("scon_localfullname2")))
            .otherwise(full_name_sort_match_key_v2_udf(F.col("scon_englishfirstname"), F.col("scon_englishlastname"), F.col("scon_englishfullname"),
                                                    F.col("scon_localfirstname"), F.col("scon_locallastname"), F.col("scon_localfullname"))))
        .withColumn("FullNameMatchKey", F.col("name_key.asc"))
        .withColumn("FullNameMatchKeyDesc", F.col("name_key.desc"))
        .drop("name_key", "scad_city_localdesc", "scad_postalcode")
    )
    

    return merged_df

#2.1 vertices,edge generate

In [0]:
def generate_vertices_and_edge(df_with_id):

    # 差异 2: 所有规则增加 OtherBlockingKey 一致性约束
    # 确保 EmpBrand99 只与 EmpBrand99 匹配，DrJart 只与 DrJart 匹配，Normal 只与 Normal 匹配
    _bk = (F.col("a.OtherBlockingKey") == F.col("b.OtherBlockingKey"))
    _mk = (F.col("a.scon_mrkt_code") == F.col("b.scon_mrkt_code"))

    # 部分market启用localname2进行匹配
    _llname2c = (F.col("a.scon_mrkt_code").isin(MARKETS_ENABLE_LOCALNAME2))


    # EnglishFName 通过 编辑距离算法 计算文本相似度, 相似度>=0.85 则通过match
    common_english_name_cond = ( 
        (F.col("a.EnglishLNameMatchKey") == F.col("b.EnglishLNameMatchKey")) & 
        
        (F.when(F.greatest(F.length("a.EnglishFNameMatchKey"), F.length("b.EnglishFNameMatchKey")) == F.lit(0), F.lit(1.0)) 
            .otherwise( F.lit(1) - (F.levenshtein(F.col("a.EnglishFNameMatchKey"), F.col("b.EnglishFNameMatchKey")) / F.greatest(F.length("a.EnglishFNameMatchKey"), F.length("b.EnglishFNameMatchKey"))))  >= 0.85
        ) &

        (F.concat(F.col("a.EnglishLNameMatchKey"), F.col("b.EnglishLNameMatchKey"), F.col("a.EnglishFNameMatchKey"), F.col("b.EnglishFNameMatchKey")) != F.lit("")) 
    )
    # (F.col("a.EnglishFNameMatchKey") == F.col("b.EnglishFNameMatchKey")) & 

    common_local_name_cond = (
        (F.col("a.LocalFNameMatchKey") == F.col("b.LocalFNameMatchKey")) & (F.col("a.LocalLNameMatchKey") == F.col("b.LocalLNameMatchKey")) &
        (F.concat(F.col("a.LocalFNameMatchKey"), F.col("b.LocalFNameMatchKey"), F.col("a.LocalLNameMatchKey"), F.col("b.LocalLNameMatchKey")) != F.lit(""))
    )

    common_local_name2_cond = (
        (F.col("a.LocalFNameMatchKey2") == F.col("b.LocalFNameMatchKey2")) & (F.col("a.LocalLNameMatchKey2") == F.col("b.LocalLNameMatchKey2")) &
        (F.concat(F.col("a.LocalFNameMatchKey2"), F.col("b.LocalFNameMatchKey2"), F.col("a.LocalLNameMatchKey2"), F.col("b.LocalLNameMatchKey2")) != F.lit(""))
    )

    # 规则 A: Email + Name
    email_cond_base = _bk & _mk & (F.col("a.EmailMatchKey") == F.col("b.EmailMatchKey"))
    condA1 = email_cond_base & (F.col("a.FullNameMatchKey") == F.col("b.FullNameMatchKey"))
    condA2 = email_cond_base & common_english_name_cond
    condA3 = email_cond_base & common_local_name_cond
    condA4 = email_cond_base & _llname2c & common_local_name2_cond
    condA5 = email_cond_base & (F.col("a.FullNameMatchKeyDesc") == F.col("b.FullNameMatchKeyDesc"))


    # 规则 B: Phone + Name
    phone_cond_base = _bk & _mk & (F.col("a.PhoneMatchKey") == F.col("b.PhoneMatchKey"))
    condB1 = phone_cond_base & (F.col("a.FullNameMatchKey") == F.col("b.FullNameMatchKey"))
    condB2 = phone_cond_base & common_english_name_cond
    condB3 = phone_cond_base & common_local_name_cond
    condB4 = phone_cond_base & _llname2c & common_local_name2_cond
    condB5 = phone_cond_base & (F.col("a.FullNameMatchKeyDesc") == F.col("b.FullNameMatchKeyDesc"))


    # 规则 C: Address + Name
    address_cond_base =  _bk & _mk & (F.col("a.AddressMatchKey") == F.col("b.AddressMatchKey"))
    condC1 = address_cond_base & (F.col("a.FullNameMatchKey") == F.col("b.FullNameMatchKey"))
    condC2 = address_cond_base & common_english_name_cond
    condC3 = address_cond_base & common_local_name_cond
    condC4 = address_cond_base & _llname2c & common_local_name2_cond
    condC5 = address_cond_base & (F.col("a.FullNameMatchKeyDesc") == F.col("b.FullNameMatchKeyDesc"))


    # 规则 D: ConsumerID
    condD = _bk & _mk & (F.col("a.scon_brnd_code") == F.col("b.scon_brnd_code")) & (F.col("a.scon_srcs_code") == F.col("b.scon_srcs_code")) & (F.col("a.scon_consumerid") == F.col("b.scon_consumerid")) 

    # 规则 E: Master consumermdmkey
    condE = _bk & _mk & ( 
        (F.col("a.is_master_recode") == True) & (F.col("b.is_master_recode") == True) &
        (F.col("a.consumermdmkey") == F.col("b.consumermdmkey")) &
        (F.col("a.consumermdmkey").isNotNull())
    )
   
    match_types_col_1 = (
        F.when(condA1, F.lit("FullName_Email_Match"))
        .when(condA3, F.lit("LocalName_Email_Match"))
        .when(condA4, F.lit("LocalName2_Email_Match"))
        .when(condA5, F.lit("FullNameDesc_Email_Match"))
        .when(condA2, F.lit("EnglishName_Email_Match"))
    )

    match_types_col_2 = (
        F.when(condB1, F.lit("FullName_Phone_Match"))
        .when(condB3, F.lit("LocalName_Phone_Match"))
        .when(condB4, F.lit("LocalName2_Phone_Match"))
        .when(condB5, F.lit("FullNameDesc_Phone_Match"))
        .when(condB2, F.lit("EnglishName_Phone_Match"))
    )

    match_types_col_3 = (
        F.when(condC1, F.lit("FullName_Address_Match"))
        .when(condC3, F.lit("LocalName_Address_Match"))
        .when(condC4, F.lit("LocalName2_Address_Match"))
        .when(condC5, F.lit("FullNameDesc_Address_Match"))
        .when(condC2, F.lit("EnglishName_Address_Match"))
    )



    full_cond_1 = (F.col("a.id") < F.col("b.id")) & ((condA1)|(condA3)|(condA4)|(condA5) |(condA2))
    full_cond_2 = (F.col("a.id") < F.col("b.id")) & ((condB1)|(condB3)|(condB4)|(condB5) |(condB2))
    full_cond_3 = (F.col("a.id") < F.col("b.id")) & ((condC1)|(condC3)|(condC4)|(condC5) |(condC2))
    full_cond_4 = (F.col("a.id") < F.col("b.id")) & (condD)
    full_cond_5 = (F.col("a.id") < F.col("b.id")) & (condE)


    edges_1 = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond_1, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),
                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                match_types_col_1.alias("match_types")
            ).distinct()
    )

    edges_2 = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond_2, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),
                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                match_types_col_2.alias("match_types")
            ).distinct()
    )

    edges_3 = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond_3, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),
                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                match_types_col_3.alias("match_types")
            ).distinct()
    )

    edges_4 = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond_4, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),
                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                F.lit("ConsumerId_Brand_Source_Match").alias("match_types")
            ).distinct()
    )

    edges_5 = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond_5, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),
                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                F.lit("Master_Consumermdmkey_Match").alias("match_types")
            ).distinct()
    )

    edges = edges_1.unionByName(edges_2).unionByName(edges_3).unionByName(edges_4).unionByName(edges_5) \
        .groupBy("src", "dst", "src_mrkt_code", "src_srcc_id", "dst_mrkt_code", "dst_srcc_id") \
        .agg(F.array_join(F.collect_set(F.col("match_types")), ",").alias("match_types"))

    vertices = df_with_id.select(F.col("id")).distinct()
    
    return vertices, edges

  

#2.2 graph generate

In [0]:

def generate_gid(vertices, edges, df_with_id):

    g = GraphFrame(vertices, edges)
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer')}/{task_id}")
    print_log(f"tCheckpointDir: {spark.sparkContext.getCheckpointDir()}")
    
    components = g.connectedComponents(algorithm="graphx")

    target_df = df_with_id.join(
        components,
        on="id", how="left"
    )

    grp_size_df = target_df.groupBy("component").agg(F.countDistinct("scon_srcc_id").alias("grp_size"))

    final_df = target_df.join(grp_size_df, on="component", how="left").select(
        F.col("scon_srcc_id").alias("srcc_id"),
        F.col("scon_mrkt_code").alias("mrkt_code"),
        F.col("scon_brnd_code").alias("brnd_code"),
        F.col("scon_srcs_code").alias("source_code"),
        F.col("scon_consumerid").alias("consumer_id"),
        F.col("scon_sourcetimestamp").alias("source_timestamp"),
        F.col("scon_id").alias("master_scon_id"),
        F.col("consumermdmkey").alias("master_consumermdmkey"),
        F.col("master_recode_create_time"),
        F.col("is_master_recode"),
        F.col("component").alias("gid"),
        F.col("grp_size"),
        F.col("batch_id")
    ) \
    .distinct() \
    .withColumn("matc_id", F.expr("uuid()")) \
    .withColumn("task_id", F.lit(task_id)) \
    .withColumn("match_type", F.lit(MATCH_TYPE_REGULAR_STR)) \
    .withColumn("creation_dt", F.current_timestamp()) 

    return final_df

#3 new Ukey generate

In [0]:
def generate_new_ukey(final_df):

    #  is_history_flag 判断条件是为了兼容history数据 与 新数据 中scon_id生成逻辑不同导致的排序异常, 保证history数据 排序优先级高于 新数据
    #  history数据 scon_id 为单调递增数值, 因此scon_id越小代表数据越早
    #  新数据 scon_id 为uuid, 因此需要通过 recode_create_time 判断最早数据
    earliest_master_ukey = (
        final_df
        .filter(F.col("is_master_recode") == True)
        .withColumn("is_history_flag", F.when(F.length(F.col("master_scon_id")) < F.lit(36), F.lit(1)).otherwise(F.lit(0)))
        .withColumn("master_recode_create_time", F.when(F.col("is_history_flag") == F.lit(1), F.to_timestamp(F.lit("1900-01-01"))).otherwise(F.col("master_recode_create_time")))
        .withColumn("earliest_rank", F.row_number().over(
            Window.partitionBy("gid").orderBy(
                F.col("is_history_flag").desc(),
                F.col("master_recode_create_time").asc(),
                F.col("master_scon_id").asc()
            )
        ))
        .filter(F.col("earliest_rank") == 1)
        .select(F.col("gid"), F.col("master_consumermdmkey").alias("new_consumermdmkey"))
    )

    group_ukey = (
        final_df.select("gid").distinct()
        .join(earliest_master_ukey, ["gid"], "left")
        .withColumn("new_consumermdmkey",
            F.when(F.col("new_consumermdmkey").isNull(), F.expr("uuid()"))
            .otherwise(F.col("new_consumermdmkey")))
    )

    final_with_newUkey_df = final_df.join(group_ukey, ["gid"], "left")

    return final_with_newUkey_df